## v18 = targeted hybrid corpus (dgxchen 0.84 base + our solver-verified additive traces)

v17 (faithful kuangyicheng-0.88 replication) scored **0.58** — a regression. Root cause (verified by
near-controlled on-disk comparison): v13 (0.84) and v17 (0.58) share architecture EXACTLY; the only
major variable was DATA. v17's 'real x2' supervision was reasoning-FREE (boilerplate + bare answers),
which suppresses the base model's reasoning. dgxchen's data (v13=0.84) is real multi-step CoT.

**v18 = dgxchen's proven 0.84 corpus + our 100%-solver-verified traces, reformatted byte-identical**
to v13's record builder (no v14 dual-template mix). Single rich-CoT per record (NO x2 doubling — the
doubling is what sank v17). Built by `scripts/build_v18_corpus.py`:
- 7,830 dgxchen verbatim + 506 ours transformation_rules (dgxchen has 0) + 1,975 ours-only verified
  problems = **10,311 records**. All additions answer==gold verified. Coverage balanced vs test set.

**Infra = v17's PROVEN pipeline, UNCHANGED:** read-only fix (no kernel_sources), GPU fail-fast,
Unsloth bf16, r32/a32/8-targets, fp32 LoRA cast, batch4/accum8 (eff batch 32), max_grad_norm 1e9,
adam b2 0.95, stratified-by-type batching, disk discipline (adapter->/tmp, zip->/kaggle/working),
loss+grad_norm in heartbeat. ONLY the data source changed: inline-generation -> mounted v18 corpus.

**Train-time WARNING:** rich CoT (mean ~1,229 tok/record) is far longer than v17's boilerplate (~75
tok). v17's 22.6s/step does NOT transfer. ~10,311 records -> ~323 steps; at v13's ~102s/step rich
rate ~9.1h. MUST run the SMOKE (max_steps=10) first to measure steady per-step before the real run.

License: dgxchen base Apache-2.0; our additions + methodology original (CC0). Solver-verified.

## Mode Selection

In [1]:
# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")


# ============================================================
# SMOKE_TEST: validate pipeline before committing to real run
# (per PLAN.md discipline rule #6 — added 2026-05-19)
# When True: 20 records, max_steps=3, ~5-10 min runtime
# When False: full dgxchen training, ~80-90 min runtime
# ============================================================
SMOKE_TEST = True   # REAL RUN: full 10,311 rich-CoT records -> ~323 steps @ ~94-98s = ~8.5h on RTX (batch1/accum32 = eff batch 32, v13-proven config)

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/dgxchen/trained-adapter"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})


{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0, 'PRETRAINED_ADAPTER_DATASET_PATH': '/kaggle/input/datasets/dgxchen/trained-adapter'}


In [2]:
# === GPU GUARD: abort fast if not on RTX (30B bf16 needs 96GB) ===
import torch
_v = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0.0
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'} | VRAM {_v:.1f} GB")
assert _v >= 70, f"VRAM {_v:.1f}GB too small. Pick RTX Pro 6000 (after attaching the competition), Run All."
print("GPU OK.")


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102.0 GB
GPU OK.


## Setup & Model Loading

In [3]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec：", importlib.util.find_spec("triton"))


Found Triton wheels: ['/kaggle/input/datasets/llkh0a/rtx-wheels/wheels/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/llkh0a/rtx-wheels/wheels/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Custom target added: /kaggle/working/pydeps
triton spec： ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7cb0f7f63fb0>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [4]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


Training environment fixes applied.


In [5]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.


In [6]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    # GPU FAIL-FAST (eliminates the "ran on the wrong GPU" waste from v16): bf16 30B needs ~60 GB just
    # to load + training overhead, so only the RTX Pro 6000 (96 GB) works. Abort BEFORE the expensive
    # model load if a smaller accelerator was selected in the UI.
    _vram = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())) / 1e9
    _names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    print(f"GPU(s): {_names} | total VRAM {_vram:.0f} GB")
    if _vram < 70:
        raise RuntimeError(f"Total VRAM {_vram:.0f} GB too small for bf16 30B. Select RTX Pro 6000 (96 GB) "
                           f"under Settings -> Accelerator, then Save & Run All.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


Found mamba wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Found causal-conv1d wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
Torch CUDA: 12.8
GPU(s): ['NVIDIA RTX PRO 6000 Blackwell Server Edition'] | total VRAM 102 GB


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Selected causal wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Selected mamba wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.


In [7]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-06-05 20:36:46.729186: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780691806.895530      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780691806.942549      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780691807.368037      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780691807.368052      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780691807.368053      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded with Unsloth.


In [8]:
if TRAIN_ON_KAGGLE:
    import sys, torch
    from unsloth import FastLanguageModel

    LORA_RANK = 32
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.0
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj", "up_proj", "down_proj",
    ]

    print("Creating trainable LoRA via FastLanguageModel.get_peft_model ...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

    # --- kuangyicheng delta #3: enable Mamba CUDA fast path (try/except for safety) ---
    try:
        _nm = None
        for _n, _m in sys.modules.items():
            if "modeling_nemotron_h" in _n and hasattr(_m, "is_fast_path_available"):
                _nm = _m; break
        if _nm:
            _nm.is_fast_path_available = True
            print("Mamba fast path enabled")
        else:
            print("Mamba module not found for fast-path patch (Unsloth may handle it)")
    except Exception as e:
        print(f"Mamba fast path skipped: {e}")

    # --- kuangyicheng delta #1: cast LoRA params to fp32 (base stays bf16) for precision ---
    _n_cast = 0
    for name, param in model.named_parameters():
        if ".lora_" in name:
            param.data = param.data.to(torch.float32); _n_cast += 1
    print(f"Cast {_n_cast} LoRA tensors to fp32 (base remains bf16)")

    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping trainable LoRA construction.")

Creating trainable LoRA via FastLanguageModel.get_peft_model ...
Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
Mamba fast path enabled
Cast 12008 LoRA tensors to fp32 (base remains bf16)
trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


## Mode A: Train on Kaggle

In [9]:
if TRAIN_ON_KAGGLE:
    import os, json, math, time, random
    import numpy as np
    import pandas as pd
    import torch
    from collections import defaultdict
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig
    from torch.utils.data import DataLoader, Sampler

    SEED = 3407
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    # ---- v18 corpus: pre-built, mounted as a Kaggle dataset (NOT generated inline) ----
    # Records are ALREADY in v13 format: messages=[user(prompt+suffix), assistant(cot + </think> + \boxed{ans})]
    # plus a 'type' label for stratified batching. Built by scripts/build_v18_corpus.py.
    import glob
    _cands = glob.glob('/kaggle/input/**/v18-corpus-filtered075.jsonl', recursive=True)
    print('corpus candidates found:', _cands)
    if not _cands:
        raise FileNotFoundError('v18-corpus-filtered075.jsonl not found under /kaggle/input — attach the '
                                'gideonnazarian/nemotron-v18-corpus-filtered075 dataset to this kernel.')
    CORPUS = _cands[0]

    all_records = []
    with open(CORPUS) as fh:
        for line in fh:
            line = line.strip()
            if line: all_records.append(json.loads(line))

    # SMOKE: stratified-ish small subset (keep type variety so the sampler is exercised)
    if SMOKE_TEST:
        random.Random(SEED).shuffle(all_records)
        all_records = all_records[:400]

    records = [{'messages': r['messages']} for r in all_records]
    record_types = [r.get('type', 'other') for r in all_records]

    dataset = HFDataset.from_list(records)
    print(f'v18 records: {len(records)} | source={CORPUS} | SMOKE={SMOKE_TEST}')
    print('By type:', dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

    def formatting_prompts_func(example):
        messages = example['messages']
        conversations = [messages] if (messages and isinstance(messages[0], dict)) else messages
        texts = []
        for conv in conversations:
            try:
                text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False, enable_thinking=True)
            except TypeError:
                text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        return texts

    training_args = SFTConfig(
        output_dir='/tmp/sft_output',
        num_train_epochs=1,
        max_steps=10 if SMOKE_TEST else -1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=32,
        learning_rate=2e-4,
        lr_scheduler_type='linear',
        warmup_steps=0,
        max_length=MAX_SEQ_LEN,
        adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
        weight_decay=0.0,
        max_grad_norm=1e9,
        logging_steps=1,
        save_strategy='no',
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        report_to='none',
        packing=False,
    )

    # ---- stratified batching by type (kuangyicheng / nemotron-master) ----
    def build_stratified_index_order(labels, batch_size, seed):
        by_label = defaultdict(list)
        for idx, label in enumerate(labels): by_label[label].append(idx)
        r = random.Random(seed)
        for v in by_label.values(): r.shuffle(v)
        n_batches = max(1, math.ceil(len(labels) / batch_size))
        batches = [[] for _ in range(n_batches)]
        order = list(range(n_batches)); r.shuffle(order)
        i = 0
        for label in sorted(by_label.keys()):
            for idx in by_label[label]:
                batches[order[i % n_batches]].append(idx); i += 1
        out = [idx for b in batches for idx in b]
        if len(out) != len(labels): raise ValueError('stratified size mismatch')
        return out

    class PrecomputedOrderSampler(Sampler):
        def __init__(self, order): self.order = list(order)
        def __iter__(self): return iter(self.order)
        def __len__(self): return len(self.order)

    class StratifiedSFTTrainer(SFTTrainer):
        def __init__(self, *args, stratified_order=None, **kwargs):
            super().__init__(*args, **kwargs); self.stratified_order = stratified_order
        def get_train_dataloader(self):
            if self.stratified_order is None: return super().get_train_dataloader()
            kw = {'batch_size': self.args.per_device_train_batch_size,
                  'sampler': PrecomputedOrderSampler(self.stratified_order),
                  'collate_fn': self.data_collator, 'num_workers': self.args.dataloader_num_workers,
                  'pin_memory': self.args.dataloader_pin_memory,
                  'persistent_workers': self.args.dataloader_persistent_workers,
                  'drop_last': self.args.dataloader_drop_last}
            if self.args.dataloader_num_workers > 0: kw['prefetch_factor'] = self.args.dataloader_prefetch_factor
            return DataLoader(self.train_dataset, **kw)

    eff_bs = max(1, training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
    strat_order = build_stratified_index_order(record_types, eff_bs, SEED)
    print(f'Effective batch: {eff_bs}, steps/epoch ~= {math.ceil(len(records)/eff_bs)}')

    # ---- heartbeat (granular progress; stuck if dt_last > 5 min) ----
    from transformers import TrainerCallback
    class _Heartbeat(TrainerCallback):
        def __init__(self): self.last = time.time(); self.start = time.time()
        def on_log(self, args, state, control, logs=None, **kwargs):
            now = time.time(); dt = now - self.last; el = (now - self.start)/60.0
            _lg = logs or {}; _loss = _lg.get('loss'); _gn = _lg.get('grad_norm')
            try:
                mem = torch.cuda.memory_allocated()/1e9; peak = torch.cuda.max_memory_allocated()/1e9
                print(f'[HEARTBEAT] step={state.global_step}/{state.max_steps if state.max_steps>0 else "?"} '
                      f'dt_last={dt:.1f}s elapsed={el:.1f}m vram={mem:.1f}/{peak:.1f}GB loss={_loss} grad_norm={_gn}', flush=True)
            except Exception as e:
                print(f'[HEARTBEAT] step={state.global_step} dt={dt:.1f}s loss={_loss} (mem read failed: {e})', flush=True)
            self.last = now

    # plain CE (NO boxed-loss): faithful kuangyicheng
    trainer = StratifiedSFTTrainer(
        model=model, args=training_args, train_dataset=dataset,
        processing_class=tokenizer, formatting_func=formatting_prompts_func,
        stratified_order=strat_order, callbacks=[_Heartbeat()],
    )

    print(f'Starting SFT (SMOKE={SMOKE_TEST}, max_steps={"10" if SMOKE_TEST else "full epoch"}) ...')
    t0 = time.time(); trainer.train(); print(f'Training done in {(time.time()-t0)/60:.1f} min')
    print(f'Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB')

    # DISK FIX: save adapter to /tmp (off the 20 GB /kaggle/working cap). Only the zip goes to /kaggle/working.
    ADAPTER_DIR = '/tmp/v17_adapter'
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f'Adapter saved to {ADAPTER_DIR}')
    import subprocess as _sp
    _sp.run(['du','-sh',ADAPTER_DIR]); _sp.run(['df','-h','/tmp','/kaggle/working'])
else:
    print('USE_PRETRAINED path: see Mode B.')

corpus candidates found: ['/kaggle/input/datasets/gideonnazarian/nemotron-v18-corpus-filtered075/v18-corpus-filtered075.jsonl']
v18 records: 400 | source=/kaggle/input/datasets/gideonnazarian/nemotron-v18-corpus-filtered075/v18-corpus-filtered075.jsonl | SMOKE=True
By type: {'bit_manipulation': 95, 'cipher': 81, 'equation_numeric_deduce': 31, 'equation_numeric_guess': 5, 'gravity': 65, 'numeral': 45, 'unit_conversion': 78}
Effective batch: 32, steps/epoch ~= 13


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/400 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting SFT (SMOKE=True, max_steps=10) ...
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,16.693800
2,16.157000
3,15.098800
4,13.219700
5,11.764600
6,11.608600
7,11.037500
8,9.930300
9,9.412400
10,9.333900


[HEARTBEAT] step=1/10 dt_last=164.8s elapsed=2.7m vram=68.6/82.4GB loss=16.6938 grad_norm=4.209048271179199
[HEARTBEAT] step=2/10 dt_last=97.6s elapsed=4.4m vram=68.6/84.4GB loss=16.157 grad_norm=3.960555076599121
[HEARTBEAT] step=3/10 dt_last=98.9s elapsed=6.0m vram=68.6/84.4GB loss=15.0988 grad_norm=3.41825270652771
[HEARTBEAT] step=4/10 dt_last=98.1s elapsed=7.7m vram=68.6/84.4GB loss=13.2197 grad_norm=2.6680026054382324
[HEARTBEAT] step=5/10 dt_last=102.2s elapsed=9.4m vram=68.6/84.4GB loss=11.7646 grad_norm=1.677034616470337
[HEARTBEAT] step=6/10 dt_last=100.8s elapsed=11.0m vram=68.6/84.4GB loss=11.6086 grad_norm=1.6888225078582764
[HEARTBEAT] step=7/10 dt_last=99.2s elapsed=12.7m vram=68.6/84.4GB loss=11.0375 grad_norm=1.7814515829086304
[HEARTBEAT] step=8/10 dt_last=100.3s elapsed=14.4m vram=68.6/84.4GB loss=9.9303 grad_norm=1.6312867403030396
[HEARTBEAT] step=9/10 dt_last=100.8s elapsed=16.0m vram=68.6/84.4GB loss=9.4124 grad_norm=1.5553940534591675
[HEARTBEAT] step=10/10 dt_l

## Mode B: Load Pre-trained LoRA

In [10]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.


## Create submission.zip

In [11]:
# Submission packaging (disk-safe: adapter from /tmp, zip -> /kaggle/working). Runs in smoke too,
# so the save/package path is always validated (PLAN.md disk-discipline rule).
import json, os, shutil, zipfile, struct, re

OUTPUT_DIR = "/kaggle/working"
if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/tmp/v17_adapter"
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
print("Packaging adapter from:", src_adapter_dir)

required_files = ["adapter_config.json", "adapter_model.safetensors"]
SUBMISSION_ADAPTER_DIR = "/tmp/v17_submission"
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)
for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, os.path.join(SUBMISSION_ADAPTER_DIR, fname))
    print(f"Copied {fname} ({os.path.getsize(src)/1024/1024:.1f} MB)")

# sanity: report the saved representation (expect separate-expert, ~12,008 tensors — grader-compatible,
# matches the official reference and kuangyicheng's 0.88).
stp = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_model.safetensors")
with open(stp, 'rb') as f:
    hlen = struct.unpack('<Q', f.read(8))[0]; hdr = json.loads(f.read(hlen))
names = [k for k in hdr if k != '__metadata__']
print(f"saved adapter tensors: {len(names)} | has_experts={any('expert' in n for n in names)}")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0
with open(config_path, "w") as f: json.dump(cfg, f, indent=2)
print(f"config target_modules: {cfg.get('target_modules')}")

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        zf.write(os.path.join(SUBMISSION_ADAPTER_DIR, fname), fname)
with zipfile.ZipFile(zip_path) as zf:
    bad = zf.testzip()
print("zip CRC OK" if bad is None else f"BAD zip: {bad}")
print(f"submission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
import subprocess as _sp; _sp.run(['du','-sh',OUTPUT_DIR]); _sp.run(['df','-h',OUTPUT_DIR])
if SMOKE_TEST:
    print("\n[SMOKE] data + fp32 cast + train + save + zip all validated. Set SMOKE_TEST=False (Mode Selection), re-run all.")
else:
    print("\n=== REAL RUN done. Submit /kaggle/working/submission.zip via Output tab -> Submit to Competition. ===")

Packaging adapter from: /tmp/v17_adapter
Copied adapter_config.json (0.0 MB)
Copied adapter_model.safetensors (3373.4 MB)
saved adapter tensors: 12008 | has_experts=True
config target_modules: ['down_proj', 'k_proj', 'o_proj', 'up_proj', 'v_proj', 'out_proj', 'q_proj', 'in_proj']
zip CRC OK
submission.zip: 3066.9 MB
3.7G	/kaggle/working
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  3.7G   16G  19% /kaggle/working

[SMOKE] data + fp32 cast + train + save + zip all validated. Set SMOKE_TEST=False (Mode Selection), re-run all.
